# Models

This notebook contains work done on the models, including code and logs of running each component model and metrics.

Functional pieces of code remain in their respective Python modules and are imported here when necessary.

## Running the code

As per the Dataset Audit notebook, the following shell commands need to be run in the `models` directory to be able to run the code:

```bash
# Set up virtual environment
python -m venv venv

# Install required packages
pip install -r requirements.txt

# Set up kernel
python -m ipykernel install --user --name=<Jupyter_Kernel_Name>
```

This should be followed by selecting the kernel in the Jupyter notebook.

## Dataset loading and preparation

To be able to load and prepare the dataset, we need to import modules from the `dataset` directory,
which is in the current folder's parent directory.

To address this, we can add the parent directory to the current PATH:

In [1]:
import os
import sys

parent = os.path.dirname(os.getcwd())
sys.path.insert(0, parent)

With this done, we can load the dataset, similarly to what we did in the Dataset Audit, 
and then feed it into the preprocessor and splitter pipeline to get the expected format.

In [3]:
import dataset.dataset_loader
from dataset.preprocessor import DatasetPipeline

corpus = dataset.dataset_loader.load_brown_corpus_df()
dataset = DatasetPipeline(corpus).get()

dataset

,raw_text,label,subset,fold_id,tokenized_text,tokenized_tags
0,"When/wrb I/ppss asked/vbd him/ppo what/wdt ,/,...",adventure,test,-1,"[When, I, asked, him, what, ,, if, anything, ,...","[wrb, ppss, vbd, ppo, wdt, ,, cs, pn, ,, ppss,..."
1,The/at guerrilla/nn bivouac/nn remained/vbd si...,adventure,test,-1,"[The, guerrilla, bivouac, remained, silent, .]","[at, nn, nn, vbd, jj, .]"
2,It/pps sprang/vbd from/in a/at type/nn of/in m...,adventure,test,-1,"[It, sprang, from, a, type, of, mentality, I'd...","[pps, vbd, in, at, nn, in, nn, ppss, vbn, rb, ..."
3,But/cc that/dt had/hvd been/ben a/at time/nn b...,adventure,test,-1,"[But, that, had, been, a, time, before, all, t...","[cc, dt, hvd, ben, at, nn, in, abn, dt, ,, at,..."
4,The/at slight/jj flutter/nn that/wps had/hvd d...,adventure,test,-1,"[The, slight, flutter, that, had, disturbed, t...","[at, jj, nn, wps, hvd, vbn, at, nn, in, ppo, n..."
...,...,...,...,...,...,...
56665,"All/ql over/in the/at country/nn ,/, belly/nn ...",editorial,train,4,"[All, over, the, country, ,, belly, clubs, hav...","[ql, in, at, nn, ,, nn, nns, hv, rb, ben, jjr,..."
56666,"When/wrb she/pps agrees/vbz ,/, you/ppss can/m...",editorial,train,4,"[When, she, agrees, ,, you, can, rest, assured...","[wrb, pps, vbz, ,, ppss, md, vb, vbn, pp$, nn,..."
56667,"He/pps should/md be/be ,/, as/cs Dag/np Hammar...",editorial,train,4,"[He, should, be, ,, as, Dag, Hammarskjold, cer...","[pps, md, be, ,, cs, np, np, rb, bedz, ,, at, ..."
56668,Possibly/rb responsible/jj for/in this/dt is/b...,editorial,train,4,"[Possibly, responsible, for, this, is, the, in...","[rb, jj, in, dt, bez, at, jj, nn, in, jj, nns,..."


In the dataframe returned above, we can see what the raw text is successfully tokenised (as per the dataset audit conclusion),
and each row is assigned to either the test set, or one of 5 folds in our $K$-fold cross-validation schema training set.

With the dataset in this shape, we can start looking at the models.

## Baseline Model

For the baseline model, as proposed previously, this project uses a majority-class tagger,
i.e. each word gets assigned the tag that it most frequently appears as in the training data.

We can use the class `MajorityTagger` I created in the `majority_tagger.py` module next to this notebook.

The `.fit` method of the `MajorityTagger` class takes in a dataset, and allows
excluding a specific fold for the cross-validation design.

We can go through the dataset excluding each of the 5 folds and gather the accuracy and macro-F1 scores for each.

In [4]:
from majority_tagger import MajorityTagger
from metrics import Metrics

baseline = MajorityTagger()
metrics = Metrics(baseline, dataset)

accuracy = []
macro_f1 = []

for fold in range(5):
    baseline.fit(dataset, fold)
    acc, _, _ = metrics.get_val_accuracy(fold, quiet=True)
    accuracy.append(acc)
    mf1, _ = metrics.get_val_macro_f1(fold, quiet=True)
    macro_f1.append(mf1)

print("Accuracy scores:", accuracy)
print("Macro-F1 scores:", macro_f1)

Accuracy scores: [0.9131985924096067, 0.9129809960962636, 0.9130836578473636, 0.9126332023924036, 0.9115693493519356]
Macro-F1 scores: [0.8390233876987407, 0.8230053656990356, 0.8371163131153176, 0.8245206483820302, 0.8358262348637081]


With these scores, we can calculate the mean and standard deviation across all folds using Python's built-in statistics module:

In [5]:
import statistics

print(f"Mean accuracy: {statistics.mean(accuracy)}, with standard deviation: {statistics.stdev(accuracy)}")
print(f"Mean Macro-F1: {statistics.mean(macro_f1)}, with standard deviation: {statistics.stdev(macro_f1)}")

Mean accuracy: 0.9126931596195146, with standard deviation: 0.0006628245233845069
Mean Macro-F1: 0.8318983899517665, with standard deviation: 0.007532217995332652


From these results, we can observe that while the accuracy is reasonably high (around 91%) with a very low standard deviation,
the macro-F1 score is noticably lower (at only about 84%) with a standard deviation about an order of magnitude higher.

This supports the dataset audit's hypothesis that due to the long-tailed, outlier-heavy distribution,
accuracy won't be a metric sufficient by itself.

Next, we can fit the baseline model to the entirety of the train set and see the overall consolidated model metrics
on the separate test set.

In [6]:
baseline.fit(dataset)

accuracy, _, _ = metrics.get_test_accuracy()
macro_f1, scores = metrics.get_test_macro_f1()

Metrics: accuracy 0.9140 (208941/228609 tagged correct) on test set
Metrics: Macro-F1 0.8337


The test set scores of about 92% accuracy and about 83% macro-F1 score are in line with what is expected
based on the $K$-fold cross-validation fitting shown previously.

To see where the macro-F1 score struggles, we can analyse the individual per-tag scores
(individual F1 score for each target variable class).

In [7]:
print(scores)

{'*': 0.9977802441731409, 'abl': 0.0, 'dt$': 1.0, 'hvn': 0.0, 'doz': 1.0, 'hvd*': 1.0, 'wp$': 1.0, 'md': 0.9912609238451935, 'nps$': 0.3076923076923077, 'wql': 0.0, 'bed': 1.0, ':': 0.976, 'hvz': 1.0, 'vbn': 0.7627672955974842, 'rb': 0.8616214639836687, 'rn': 0.6666666666666666, 'nil': 0.0, 'hv': 1.0, 'nrs': 0.8888888888888888, 'jjs': 0.75, '``': 1.0, "''": 1.0, 'ppl': 1.0, 'bez*': 0.75, 'bedz*': 1.0, ')': 0.9952267303102625, 'abn': 0.9536208299430431, 'cd': 0.9415041782729805, 'np$': 0.7913340935005702, 'vbg': 0.8903262900675578, 'pp$$': 0.8285714285714286, 'np': 0.9052969502407705, 'wps': 0.7464042392127176, 'od': 0.904330312185297, 'doz*': 1.0, 'uh': 0.6960352422907489, 'ppo': 0.7165376898824878, 'abx': 1.0, 'ap$': 1.0, 'do': 0.998019801980198, 'ppls': 1.0, 'hvd': 0.9745889387144993, 'hvg': 1.0, 'nns': 0.9420289855072463, 'ap': 0.7973781704189228, 'wdt': 0.9967697277341947, 'pps': 0.9312902833515748, 'vbz': 0.8288354898336414, 'ber*': 0.7692307692307693, 'rbt': 0.0, 'dtx': 0.0, '.':

While some of the less ambiguous and more frequent tags achieve a perfect F1 score of $1.0$,
some of the outlier tags remain at $0.0$ which will inevitably drag the arithmetic mean value down.

Some of the tags common with less clear-cut words also have noticably lower scores, for example `vbd`, a past-tense verb,
only achieves an F1 score of about $0.743$ (as words that are past-tense verbs in some contexts can be, for example, nouns in others).

The notably lower macro-F1 score as well as analysing the individual per-tag F1 scores indicates that
the baseline majority-class tagger is capable of dealing with some of the more common tags (which dominate this dataset) 
and overall tags the majority of them correctly, but struggles with outliers and doesn't achieve a very high score
in frequent but commonly somewhat ambiguous tags.

## Bidirectional LSTM Model

The first of the proposed models is a Bi-LSTM, a bidirectional recurrent neural network using the long short-term memory system.

TODO Add more content about LSTMs here maybe, if necessary or fitting into the doc

### Hyperparameter Tuning

Before we train the model on the full training dataset, we can tune a range of the model's hyperparameters
using the proposed K-fold cross validation strategy.

By running training for N epochs 5 times, excluding each of the 5 folds from our split once, then measuring metrics
and calculating the mean and standard deviation, we can see how changing hyperparameters affects model performance.
Based on which values lead to the highest F1 score, we can choose the optimal model settings to train with.

For the evaluation we'll use the F1 score, as it's more sensitive to the model's ability to fit to the outlier-heavy distribution.
We'll tune the learning rate, word emebedding dimension, the LSTM model's hidden dimension, the number of LSTM layers present
as well as the dropout probability.

We can use the original BiLSTMTagger class and its static method `BiLSTMTagger.tune()` to perform a tuning run,
with a given number of epochs and given parameter values.

Since training this type of model takes a fairly notable amount of time, for performance reasons we do the tuning on
a randomly sampled half of the dataset (with a specified seed for reproducibility).
Similarly for performance reasons, we choose an `N=3` to run 3 training epochs, enough to be able to observe improvement with training,
but still relatively cheap to run.

For the same performance reasons, instead of a true grid, we assume that our hyperparameters are reasonably independent,
and examine them sequentially.
We can verify that the independence assumption holds once we train a model with all the parameters combined
if the resulting accuracy is at least as good as the best candidate model during parameter selection.

The following code cell runs the hyperparameter tuning as described above.

In [10]:
from lstm_tagger import BiLSTMTagger

n_epochs = 3
ds_fraction = 0.5
folds = [0, 1, 2, 3, 4] # using all 5 folds

candidate_lr = [10**-4, 5*10**-4, 10**-3, 5*10**-3, 10**-2]
candidate_embedding = [25, 50, 100, 200]
candidate_hidden = [64, 128, 256, 512]
candidate_dropout = [0.2, 0.3, 0.4, 0.5, 0.6]
candidate_layers = [1, 2]

lstm_params = BiLSTMTagger.tune(dataset, n_epochs, ds_fraction, folds,
                                learning_rates = candidate_lr,
                                embedding_dims = candidate_embedding,
                                hidden_dims = candidate_hidden,
                                dropout_ps = candidate_dropout,
                                lstm_layers = candidate_layers)

Model Log: Hyperparameter tuning started (fraction 0.5, on 3 epochs)
Model Log: Initialised BiLSTMTagger Model on device cuda
Model Log: Starting Fitting
Model Log: Processed Sentences/Tags to IDs
Model Log: Running epoch 0...
Model Log: Epoch 0 done!
Model Score:
 -> Accuracy: 0.45810657901826607
 -> Macro-F1: 0.06272118225041044
Model Log: Running epoch 1...
Model Log: Epoch 1 done!
Model Score:
 -> Accuracy: 0.5964545503529285
 -> Macro-F1: 0.2123798096687648
Model Log: Running epoch 2...
Model Log: Epoch 2 done!
Model Score:
 -> Accuracy: 0.6524920523735115
 -> Macro-F1: 0.3083685109407054
Model Log: Initialised BiLSTMTagger Model on device cuda
Model Log: Starting Fitting
Model Log: Processed Sentences/Tags to IDs
Model Log: Running epoch 0...
Model Log: Epoch 0 done!
Model Score:
 -> Accuracy: 0.4692419589461775
 -> Macro-F1: 0.0784586078084084
Model Log: Running epoch 1...
Model Log: Epoch 1 done!
Model Score:
 -> Accuracy: 0.6066052475620926
 -> Macro-F1: 0.21494116560154852
Mo

/media/martin/data/martin/MyThings/School/AY2627/CT4101/portfolio/repo/models/venv/lib64/python3.14/site-packages/torch/nn/modules/rnn.py:1012: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Model Log: Initialised BiLSTMTagger Model on device cuda
Model Log: Starting Fitting
Model Log: Processed Sentences/Tags to IDs
Model Log: Running epoch 0...
Model Log: Epoch 0 done!
Model Score:
 -> Accuracy: 0.6728164232986691
 -> Macro-F1: 0.3813117944218223
Model Log: Running epoch 1...
Model Log: Epoch 1 done!
Model Score:
 -> Accuracy: 0.7450616951344361
 -> Macro-F1: 0.4832220746873604
Model Log: Running epoch 2...
Model Log: Epoch 2 done!
Model Score:
 -> Accuracy: 0.7814645185624225
 -> Macro-F1: 0.5686817783203847
Model Log: Initialised BiLSTMTagger Model on device cuda
Model Log: Starting Fitting
Model Log: Processed Sentences/Tags to IDs
Model Log: Running epoch 0...
Model Log: Epoch 0 done!
Model Score:
 -> Accuracy: 0.6751144873659825
 -> Macro-F1: 0.37909241641589675
Model Log: Running epoch 1...
Model Log: Epoch 1 done!
Model Score:
 -> Accuracy: 0.7484618285652712
 -> Macro-F1: 0.48052479370678813
Model Log: Running epoch 2...
Model Log: Epoch 2 done!
Model Score:
 -> 

The following code cell contains a data dump of all the information gathered during the hyperparameter search,
as well as a formatted version of the case table, displaying the mean and standard deviation obtained for each
tested parameter or combination.

In [25]:
print(lstm_params)
print("---")
print("Case Table:")
for key in lstm_params["case_table"]:
    print(f"{key}:")
    for element in lstm_params["case_table"][key]:
        stats = element[0]
        mean = element[1]
        stdev = element[2]
        print(" - ", end="")
        for stat_key in stats:
            print(f"{stat_key}: {stats[stat_key]}", end="; ")
        print(f"\n   - {mean} ({stdev})")

{'learning_rate': ({'learning_rate': 0.005}, np.float64(0.8129323371575413), np.float64(0.011192407256231869)), 'dropout_p': ({'dropout_p': 0.2}, np.float64(0.7482277935417783), np.float64(0.011595669348429028)), 'embedding_dim': ({'embedding_dim': 200}, np.float64(0.7336676795555307), np.float64(0.009199330383799158)), 'lstm_params': ({'hidden_dim': 512, 'lstm_layers': 2}, np.float64(0.7470563203259865), np.float64(0.012841860472289069)), 'case_table': {'learning_rate': [({'learning_rate': 0.0001}, np.float64(0.30709041683306604), np.float64(0.007103380615731246)), ({'learning_rate': 0.0005}, np.float64(0.5933037946619043), np.float64(0.00733238973559409)), ({'learning_rate': 0.001}, np.float64(0.6950893183779732), np.float64(0.01152970938741446)), ({'learning_rate': 0.005}, np.float64(0.8129323371575413), np.float64(0.011192407256231869)), ({'learning_rate': 0.01}, np.float64(0.8035866988952709), np.float64(0.008974360290059996))], 'dropout_p': [({'dropout_p': 0.2}, np.float64(0.7482

The results suggest that good hyperparameters are as follows:

- learning_rate: 0.005
- dropout_p: 0.2
- embedding_dim: 200
- hidden_dim: 512
- lstm_layers: 2

The limitations of this search are that the parameter count values are the maximum of the proposed range,
meaning there is a chance that increasing them further would lead to even better results.

However, for the scope of this project, we are resource and time-constrained, and, as such, searching through
even larger hidden parameter counts is not very feasible.
This fact is a limitation that should be noted in the final report.

The next cell contains a verification run, training a model on the entirety of the train set once,
using the combined hyperparameters found previously, still on 3 epochs and half of the dataset.

In [18]:
tagger = BiLSTMTagger(dataset, learning_rate=0.005, dropout_p=0.2, embedding_dim=200, hidden_dim=512, lstm_layers=2)
tagger.fit(dataset, 3, 0.5)

Model Log: Initialised BiLSTMTagger Model on device cuda
Model Log: Starting Fitting
Model Log: Processed Sentences/Tags to IDs
Model Log: Running epoch 0...
Model Log: Epoch 0 done!
Model Score:
 -> Accuracy: 0.9442642383749125
 -> Macro-F1: 0.8525287995731881
Model Log: Running epoch 1...
Model Log: Epoch 1 done!
Model Score:
 -> Accuracy: 0.9762271674120373
 -> Macro-F1: 0.8983610593189499
Model Log: Running epoch 2...
Model Log: Epoch 2 done!
Model Score:
 -> Accuracy: 0.986022953822943
 -> Macro-F1: 0.9300508693991344


(0.986022953822943, 0.9300508693991344)

We can see that combining all hyperparameters leads to a better result than any parameter individually,
supporting our assumption that for the scope of this project, each parameter stage can be treated
as additive, independent configurations.